# **IS 4487 Week 6 - Data cleaning**

This notebook is designed to help you follow along with the Week 6 Lecture and Reading, introducing you to Python.

The practice code demos are intended to give you a chance to see working code and can be a source for your lap and assignment work. Each section contains short explanations and annotated code that reflect the steps in the reading.





### **Topics for this demo:**



*   Cleaning data (handling missing values, fixing formats, removing duplicates)
*   Data validation (ensure business rule compliance with the cleaned data)

<a href="https://colab.research.google.com/github/Stan-Pugsley/is_4487_base/blob/main/Demos/demo_06_data_cleaning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# **Context:**

StreamFlow, Inc. is a fast-growing streaming service that offers curated video content to
subscribers worldwide. The marketing analytics team has been tasked with building a customer
churn prediction model to identify users at risk of canceling their subscriptions. The dataset
includes customer demographics, subscription history, usage patterns, and account activity logs.

However, before modeling can begin, the team needs to clean and prepare the raw customer
data, which was pulled from multiple internal systems (CRM, billing, and user behavior
tracking). The dataset includes inconsistencies, missing values, and mixed data types.

# Is this problem statement well-aligned with the CRISP-DM framework?

### **Import Libraries**






In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np


### **Load Dataframe**





In [ ]:
URL = 'https://raw.githubusercontent.com/Stan-Pugsley/is_4487_base/refs/heads/main/DataSets/streamflow_customer_churn.csv'
df = pd.read_csv(URL)


# the other way to load dataset?

### **Preliminary data inspection**





In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
df.head()

In [ ]:
df.info()

# How is the data type?

# Any missing values here?


In [ ]:
df.shape

In [ ]:
df.describe()

---
---
# **Cleaning data**

since our data is only 250 rows and our goal is to get it ready for machine learning, we will be dropping as few rows as possible and imputing missing data






# Which way do you like to handle missing values?

---
### **Change the columns into their proper datatypes**

In [ ]:
categorical_cols = ['gender', 'region', 'plan_type']
for col in categorical_cols:
    df[col] = df[col].astype('category')

df['signup_date'] = pd.to_datetime(df['signup_date'], errors='coerce')


In [ ]:
df.info()

---
### **Look for missing values**







In [ ]:
df.isnull().sum()

If we hadn't converted the data type, we wouldn't have known that 8 signup_date values ​​were missing.

In [ ]:
len(df[df.isnull().any(axis=1)])

In [ ]:
df[df.isnull().any(axis=1)]

---
### **Look for duplicate rows**

In [ ]:
print(f' Number of duplicate rows: {df.duplicated().sum()}')

---
### **Look for formatting issues in the categorical columns**

In [ ]:
print('Looking for typos/formatting issues in the data:')
for cats in categorical_cols:
  print(f'{cats}:  {df[cats].unique()}')

---
###**Cleaning up the missing dates**

We will fill in the missing dates from known dates in a linear order. Basically Y=mx + B where you draw a line between two known points

In [ ]:
df['signup_date'] = df['signup_date'].interpolate(method='linear')

missing_date_check = df['signup_date'].isnull().sum()
print(f"the number of missing dates is: {missing_date_check}")


---
###**Eliminating invalid ages**

Earlier when we used df.describe() we saw that we had a minimum age of -10 and a maximum of 150 and we are missing 12 values.

First lets look at the distribution of age to decide what method of imputation we should use to fill the missing values (Mean, Median, Mode)


In [ ]:
# Plot a histogram of the age distribution
plt.figure(figsize=(10, 6))
plt.hist(df['age'].dropna(), bins=20, edgecolor='black')
plt.xlabel('Age')
plt.ylabel('Frequency')
plt.title('Distribution of Age')
plt.grid(axis='y', alpha=0.75)
plt.show()

Ignoring the outliers we have a normal distribution, from here we can mark the outliers as NaN and fill in the NaN values with the median age.

We chose to median imputation because using the median is more robust than the mean and wont be as affected by remaining edge cases

In [ ]:
# Filter out ages less than 18 or greater than 120
df.loc[(df['age'] < 0) | (df['age'] > 120), 'age'] = np.nan

# Step 2: Impute all NaN ages with median
df['age'] = df['age'].fillna(df['age'].median()).round().astype(int)

---
###**filling in missing values for gender**

We have 10 missing values for gender, we could impute the data using mean/median/mode but the drawback would be that we could skew our distrubtion throwing off a machine learning model. Lets look at the distribuition first

In [ ]:
# find gender distribution
gender_dist = df['gender'].value_counts(normalize=True)
print(gender_dist)

Since the distribution is essentially 33% for each category, a mode imputation would skew the gender distribution too heavily. We could drop the rows here but, since we are trying to preserve as much data as possible we will replace the NaN values with "unknown"  

We will encode gender later to get the dataset ready for machine learning.

In [ ]:
# Add 'Unknown' as a new category to the 'gender' column
df['gender'] = df['gender'].cat.add_categories('Unknown')

# Fill missing values in the 'gender' column with 'Unknown'
df['gender'] = df['gender'].fillna('Unknown')

# Print the value counts to verify the changes
print(df['gender'].value_counts(dropna=False))

Why didn't we impute gender using some ML tools? The reason We chose not to impute missing gender values with machine learning because predicting sensitive attributes could introduce bias and create ethical concerns.

Instead, we created an "Uknown" category, which preserves all rows and allows the model to learn if missingness itself is a signal of disengagement or churn. This approach avoids false assumptions while still providing potentially valuable insights for business decision-making.

---
###**filling in missing values for plan type**

Imputing missing values for the plan type may be very intuitinve becuase we can assume that their is a relationship between plan_type and Monthly_fee.

We may be able to use regression based on monthly_fee to classify plan_types

lets look at the data by plan type and see if there is an obvoius range for plan type and monthly_fee that would allow us to impute the missing values


In [ ]:
# Group means
print(df.groupby('plan_type')['monthly_fee'].mean())

# Boxplot visualization
df.boxplot(column='monthly_fee', by='plan_type', figsize=(6,4))
plt.title("Monthly Fee by Plan Type")
plt.suptitle("")
plt.show()


Looking at this boxplot, we can see that the monthly fees for Basic, Premium, and Standard customers overlap heavily. None of the plan types has a clearly distinct fee range, which means we cannot reliably use monthly fee (or related variables) to predict the missing plan types.

Because of this overlap, the best approach is to treat missing plan types as "unknown" , allowing our machine learning model to decide if missingness itself carries predictive power.

In [ ]:
# create a new category for plan_type to for NaNs
df['plan_type'] = df['plan_type'].cat.add_categories(['Unknown'])

#Impute NaNs with "unknown"
df['plan_type'] = df['plan_type'].fillna('Unknown')


In [ ]:
df['plan_type'].unique()

In [ ]:
df['plan_type'].isnull().sum()

---
### **Final Steps**

We are going to copy the changes we made in our dataframe to a machine learning ready data frame.

We will drop the customer_id column beucase it serves no purpose for machine learning, check to make sure there are no missing values one last time, and take a quick glance at the machine learning ready dataframe

In [ ]:
# Make a copy for ML
df_ML_ready = df.copy()

# Do you think the following code will produce the same result?

In [ ]:
df_1 = df

In [ ]:
# Drop ID column
df_ML_ready = df_ML_ready.drop(columns=['customer_id'])

#check for NaN values
df_ML_ready.isnull().sum()

In [ ]:
df_ML_ready.head()


---
### **Final Note**

From here you would potentially need to:
* convert plan_type, region, and gender into string datatypes for some machine learning tools, or keep them as objects for other machine learning tools
* Encode these categorical variables so machine learning models can work with them

In this example our main goal was to show how to clean data and missing values without losing rows from the dataset.